In [ ]:
import sys
import os
sys.path = [p for p in sys.path if "ParaView" not in p]
sys.path.append("/home/haseeb/Notebooks/Parametric_DMD_cylinder_2D_modules")



import torch as pt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as mtri
from flowtorch import DATASETS
from flowtorch.data import FOAMDataloader, mask_box
from pydmd import DMD, HODMD, ParametricDMD
from ezyrb import POD, Database, RBF, GPR, Linear
import seaborn as sns


from pathlib import Path
import re
# For High Quality Visuals
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 160  # crisp visuals


In [ ]:
# Training and test parameters (parameter set P and test parameter μ*)

# Training parameter set P = {μ1, ..., μP}, i.e., Reynolds numbers
Re_list = np.array([100, 110, 120, 130, 140, 160, 170, 180, 190, 200])  # μ: training parameters

# Test parameter μ* (unseen Reynolds number)
Re_test = 150  # μ*: test parameter
# Global reference length and viscosity
L_ref = 0.1   # cylinder diameter [m]
nu = 1e-3     # kinematic viscosity [m^2/s] (example value, adjust to your case)

# Geometry parameters (from your blockMeshDict)
cylinderX = 0.2   # cylinder center x [m]
cylinderY = 0.2   # cylinder center y [m]
radius    = 0.05  # cylinder radius [m]


# Calculate reference velocities for non-dimensionalization directly from Re
U_ref_dict = {Re: Re * nu / L_ref for Re in Re_list}
U_ref_test = Re_test * nu / L_ref

# Visualization of parameter set S and test parameter p*
fig, ax = plt.subplots(figsize=(10, 3))
plt.title('Parameter $Re$', fontsize=18, pad=10)

# Plot all training parameters (S)
ax.scatter(Re_list, np.zeros_like(Re_list), 
           marker='o', s=80, color='royalblue', edgecolor='white', 
           label='Train (S)', zorder=3)

# Plot test parameter (p*)
ax.scatter(Re_test, 0, 
           marker='D', s=120, color='darkorange', edgecolor='black', 
           label='Test (p*)', zorder=4)

# Formatting
plt.xticks(np.append(Re_list, Re_test), rotation=45, fontsize=10)
plt.yticks([])
ax.axhline(0, color='lightgray', linewidth=1.5, zorder=1)   # baseline line
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(ncols=2, frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Load and prepare data

from data_loader import load_all_snapshots, load_test_parameter

# Base folder containing simulation data
base_path = os.path.expanduser("~/OpenFOAM/test_1")


# Time windows
training_window = (10.0, 15.0)  # Time range for training snapshots
future_window = (15.0, 20.0)    # Time range for future prediction
test_window = (15.0, 20.0)       # Time range for test Reynolds number

# Sampling step for downsampling time steps
sampling_step = 1

# Load training and future snapshots for all training Reynolds numbers
data = load_all_snapshots(
    Re_list=Re_list,
    base_path=base_path,
    mask_box=mask_box,
    FOAMDataloader=FOAMDataloader,
    training_window=training_window,
    future_window=future_window,
    sampling_step=sampling_step
)

# Access training data
snapshot_dict = data["snapshot_dict"]
sampled_times_dict = data["sampled_times_dict"]
snapshot_future_dict = data["snapshot_future_dict"]
sampled_times_future_dict = data["sampled_times_future_dict"]
masked_coords_dict = data["masked_coords_dict"]        
num_points_dict = data["num_points_dict"]              

# Load test snapshots for an unseen Reynolds number
path_test = f"{base_path}/cylinder_2D_Re{Re_test}"
snapshot_test, num_points_test, times_test, mask_test, coords_test, loader_test = load_test_parameter(
    Re=Re_test,
    path=path_test,
    mask_box=mask_box,
    FOAMDataloader=FOAMDataloader,
    time_window=test_window,
    sampling_step=sampling_step
)




In [ ]:
# Confirm shapes of loaded data

for Re in Re_list:
    print(f"Re={Re}, shape={snapshot_dict[Re].shape}")


In [ ]:
# Snapshot magnitude over time for each Parameter i.e., Reynolds number

from visualization_utilities import plot_snapshot_magnitudes

loader_dict = data["loader_dict"]  

plot_snapshot_magnitudes(snapshot_dict, sampled_times_dict, Re_list)


In [ ]:
# Add noise to training parameters snapshots

from noise_analysis_paramdmd import add_noise_to_snapshots

# Choose noise level(s)
noise_levels = [0, 10, 20, 40]  # or any subset you want

snapshot_noisy_dict_versions = {}

for nl in noise_levels:
    snapshot_noisy_dict_versions[nl] = add_noise_to_snapshots(
        snapshot_dict=snapshot_dict,
        Re_list=Re_list,
        noise_level=nl
    )


In [ ]:
# Visualize added noise

from noise_analysis_paramdmd import visualize_noise_levels

Re_vis = 100
noise_levels = [0, 10, 20, 40]
colors = {0: "royalblue", 10: "seagreen", 20: "darkorange", 40: "crimson"}

visualize_noise_levels(
    snapshot_noisy_dict_versions=snapshot_noisy_dict_versions,
    snapshot_clean=snapshot_dict[Re_vis],
    sampled_times_dict=sampled_times_dict,
    Re_value=Re_vis,
    noise_levels=noise_levels,
    colors=colors
)



## Preprocess training snapshots: Subtract mean flow

In [ ]:
from preprocess_snapshots import preprocess_snapshots

snapshot_noisy_processed_dict = {}
mean_flow_noisy_dict = {}   

for nl in noise_levels:
    print(f"\nPreprocessing noise level {nl}%")

    # Preprocess training datasets for noise level
    train_snapshots, mean_flow_train, snapshot_processed_dict = preprocess_snapshots(
        snapshot_noisy_dict_versions[nl],
        Re_list=Re_list
    )

    # Store processed snapshots
    snapshot_noisy_processed_dict[nl] = snapshot_processed_dict

    # Store mean flow for this noise level
    mean_flow_noisy_dict[nl] = mean_flow_train

    # Print shapes of preprocessed data
    print("Training mean flow shape:", mean_flow_train.shape)
    print("Training snapshot array shape:", train_snapshots.shape)

    for Re in Re_list:
        print(f"Noise {nl}% | Re={Re}: processed shape = {snapshot_processed_dict[Re].shape}")


## Partitioned ParametericDMD from PyDMD 

#### References

[Ichinaga et al., *PyDMD: A Python Package for Robust Dynamic Mode Decomposition*, JMLR 2024 — GitHub](https://github.com/mathLab/PyDMD)




## Offline Phase

## Step 1: Perform POD and DMD on the concatenated snapshot matrix X1

In [ ]:
# Parametric DMD

pdmd_models = {}
rom_models = {}   

for nl in noise_levels:
    print(f"\n Training PDMD for noise level {nl}% ")

    # Create shared POD basis
    rom = POD(rank=30, method="randomized_svd")

    # Create DMD instances
    trained_dmds = [DMD(svd_rank=-1) for _ in Re_list]

    # Create interpolator
    interpolator = RBF()

    # Construct ParametricDMD
    pdmd = ParametricDMD(trained_dmds, rom, interpolator)

    # Build training tensor for the noise level
    train_snapshots_noisy = np.array([
        snapshot_noisy_processed_dict[nl][Re] for Re in Re_list
    ])

    # Fit model
    pdmd.fit(train_snapshots_noisy, np.array(Re_list).reshape(-1, 1))

    # Store the model and rom
    pdmd_models[nl] = pdmd
    rom_models[nl] = rom





In [ ]:
# Visualize eigenvalue spectrum of trained DMD models

for nl in noise_levels:
    pdmd = pdmd_models[nl]   # retrieve the model for this noise level

    plt.figure(figsize=(8, 6))

    for i, dmd in enumerate(pdmd._dmd):
        eigs = dmd.eigs
        Re = Re_list[i]
        plt.scatter(np.real(eigs), np.imag(eigs),
                    label=f"$Re$ = {Re}", alpha=0.7)

    plt.xlabel("$λ_{real}$", fontsize=13)
    plt.ylabel("$λ_{im}$", fontsize=13)
    plt.title(f"Eigenvalue ($λ$) Spectrum — Noise {nl}%", fontsize=14, pad=10)
    plt.grid(True)
    plt.axis("equal")
    plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot modal coefficients True vs ParametricDMD for a training parameter i.e., Reynolds number


from visualization_utilities import plot_dmd_modal_comparison

Re_target = 160
n_modes_to_plot = 5

for nl in noise_levels:
    print(f"\nModal coefficient comparison for noise level {nl}%")

    plot_dmd_modal_comparison(
        pdmd=pdmd_models[nl],                                # PDMD for this noise level
        Re_list=Re_list,
        sampled_times_dict=sampled_times_dict,
        Re_value=Re_target,
        U_ref_dict=U_ref_dict,
        L_ref=L_ref,
        rom=rom_models[nl],                                  # ROM for this noise level
        snapshot_processed_dict=snapshot_noisy_processed_dict[nl],  # NOISY processed snapshots
        n_modes_to_plot=n_modes_to_plot
    )


In [ ]:
# Plot FFT of modal coefficients True vs ParametricDMD for a training parameter i.e., Reynolds number

from visualization_utilities import plot_dmd_fft_comparison

Re_target = 160
n_plot = 5
dt = 0.01

for nl in noise_levels:
    print(f"\nFFT comparison for noise level {nl}%")

    plot_dmd_fft_comparison(
        pdmd=pdmd_models[nl],                                 # PDMD for this noise level
        Re_list=Re_list,
        Re_target=Re_target,
        L_ref=L_ref,
        nu=nu,
        n_plot=n_plot,
        dt=dt,
        snapshot_processed_dict=snapshot_noisy_processed_dict[nl],  # NOISY processed snapshots
        rom=rom_models[nl],                                   # ROM for this noise level
    )


In [ ]:
# Plot flow comparison True vs ParametricDMD for a training parameter i.e., Reynolds number

from visualization_utilities import plot_flow_comparison_dmd_vs_true


Re_target = 160
t_start = 14.0
t_end = 15.0
granularity = 0.5

noise_level_to_plot = 40   

plot_flow_comparison_dmd_vs_true(
    Re_target=Re_target,
    Re_list=Re_list,
    t_start=t_start,
    t_end=t_end,
    granularity=granularity,
    rom=rom_models[noise_level_to_plot],                         # ROM for this noise level
    pdmd=pdmd_models[noise_level_to_plot],                       # PDMD for this noise level
    mean_flow=mean_flow_noisy_dict[noise_level_to_plot],         # mean flow for this noise level
    sampled_times_dict=sampled_times_dict,
    snapshot_dict=snapshot_dict,       
    masked_coords_dict=masked_coords_dict,
    num_points_dict=num_points_dict,
    L_ref=L_ref,
    cylinderX=cylinderX,
    cylinderY=cylinderY,
    radius=radius,
    cmap='icefire'
)


In [ ]:
# Plot reconstruction error True vs ParametricDMD for a training parameter i.e., Reynolds number

from visualization_utilities import plot_dmd_reconstruction_error

Re_target = 160

for nl in noise_levels:
    plot_dmd_reconstruction_error(
        Re_target=Re_target,
        Re_list=Re_list,
        rom=rom_models[nl],                               # ROM for this noise level
        pdmd=pdmd_models[nl],                             # PDMD for this noise level
        mean_flow_train=mean_flow_noisy_dict[nl],         # mean flow for this noise level
        sampled_times_dict=sampled_times_dict,
        snapshot_dict=snapshot_dict,     
        L_ref=L_ref,
        nu=nu,
        color='tab:orange'
    )


# Online phase

# Forecasting and interpolation

In [ ]:
for nl in noise_levels:
    print(f"\n Forecasting for noise level {nl}% ")

    pdmd = pdmd_models[nl]   # PDMD for this noise level

    # Use sampled times from the first parameter in training list
    training_times = np.array(sampled_times_dict[Re_list[0]], dtype=float)

    # Define forecasting time range
    pdmd.dmd_time["t0"]   = pdmd.original_time["tend"]
    pdmd.dmd_time["tend"] = pdmd.original_time["tend"] + 500 - 1e-12
    pdmd.dmd_time["dt"]   = pdmd.original_time["dt"]

    # Set target parameter for interpolation
    pdmd.parameters = np.array([[Re_test]])

    # Trigger interpolated reconstruction
    interpolated_snapshots = pdmd.reconstructed_data
    interpolated_field = interpolated_snapshots[0]

    # Print physical time steps
    print(
        f"Forecasting from t = {pdmd.dmd_time['t0']} "
        f"to t = {pdmd.dmd_time['tend']} with Δt = {pdmd.dmd_time['dt']}"
    )
    print("Time vector:", pdmd.dmd_timesteps)


In [ ]:
# Define forecasting time range

# Use sampled times from the first Re in your training list
training_times = np.array(sampled_times_dict[Re_list[0]], dtype=float)

# Define forecasting time range
pdmd.dmd_time["t0"]   = pdmd.original_time["tend"] 
pdmd.dmd_time["tend"] = pdmd.original_time["tend"] + 500 - 1e-12
pdmd.dmd_time["dt"]   = pdmd.original_time["dt"]


# Set target Re for interpolation
pdmd.parameters = np.array([[Re_test]])


# Trigger interpolated reconstruction 
interpolated_snapshots = pdmd.reconstructed_data  # shape: (1, space_dim, time_steps)
interpolated_field = interpolated_snapshots[0]

# Print physical time steps
print(
    f"Forecasting from t = {pdmd.dmd_time['t0']} to t = {pdmd.dmd_time['tend']} with Δt = {pdmd.dmd_time['dt']}"
)
print("Time vector:", pdmd.dmd_timesteps)



In [ ]:
# Plot reconstruction error True vs Forecasted ParametricDMD for a training parameter i.e., Reynolds number

from visualization_utilities import plot_dmd_forecast_error

Re_target = 160

for nl in noise_levels:
    print(f"\nForecast error for noise level {nl}%")

    plot_dmd_forecast_error(
        Re_target=Re_target,
        Re_list=Re_list,
        rom=rom_models[nl],                               # ROM for this noise level
        pdmd=pdmd_models[nl],                             # PDMD for this noise level
        mean_flow_train=mean_flow_noisy_dict[nl],         # mean flow for this noise level
        sampled_times_dict=sampled_times_dict,
        snapshot_future_dict=snapshot_future_dict,  
        L_ref=L_ref,
        nu=nu,
        color='tab:blue'
    )



In [ ]:
# Plot modal coefficients True vs Interpolated ParametricDMD for unseen parameter i.e., Reynolds number

from visualization_utilities import plot_dmd_modal_comparison_interp_vs_true

Re_target = 160
n_modes_to_plot = 5

for nl in noise_levels:
    print(f"\n Interpolated modal comparison for noise level {nl}% ")

    plot_dmd_modal_comparison_interp_vs_true(
        pdmd=pdmd_models[nl],                 # PDMD for this noise level
        snapshot_test=snapshot_test,    
        loader_test=loader_test,
        Re_test=Re_target,
        dt_phys=0.01,
        t0_phys=15.0,
        n_modes_to_plot=n_modes_to_plot,
        L_ref=L_ref,
        nu=nu,
        rom=rom_models[nl],                   # ROM for this noise level
        times_test=times_test
    )



In [ ]:
# Plot FFT of modal coefficients interpolated modal coefficients for unseen parameter i.e., Reynolds numbers

from visualization_utilities import plot_dmd_fft_comparison_interp_vs_true

n_modes_to_plot = 5
dt_phys = 0.01
t0_phys = 15.0

for nl in noise_levels:
    
    plot_dmd_fft_comparison_interp_vs_true(
        pdmd=pdmd_models[nl],                 # PDMD for this noise level
        snapshot_test=snapshot_test,          
        loader_test=loader_test,
        Re_test=Re_test,
        L_ref=L_ref,
        nu=nu,
        rom=rom_models[nl],                   # ROM for this noise level
        dt_phys=dt_phys,
        t0_phys=t0_phys,
        n_modes_to_plot=n_modes_to_plot,
        times_test=times_test
    )



In [ ]:
# Plot flow comparison true vs interpolated ParametricDMD for unseen parameter i.e., Reynolds number
from visualization_utilities import plot_flow_comparison_interpolated_dmd_vs_true

t_start = 18.0
t_end = 19.0
granularity = 0.5
noise_level_to_plot = 40   # choose the noise level

plot_flow_comparison_interpolated_dmd_vs_true(
    pdmd=pdmd_models[noise_level_to_plot],                      # PDMD for this noise level
    snapshot_test=snapshot_test,                    
    sampled_times_test=times_test,
    loader_test=loader_test, 
    mask_test=mask_test,
    num_points_test=num_points_test,
    mean_flow_train=mean_flow_noisy_dict[noise_level_to_plot],  # mean flow for this noise level
    Re_test=Re_test,
    t_start=t_start,
    t_end=t_end,
    granularity=granularity,
    L_ref=0.1,
    cmap="icefire"
)



In [ ]:
# Plot interpolation reconstruction error unseen parameter i.e., Reynolds number

from visualization_utilities import plot_interp_reconstruction_error

for nl in noise_levels:
    print(f"\n Interpolated reconstruction error for noise level {nl}% ")

    plot_interp_reconstruction_error(
        snapshot_test=snapshot_test,                     
        times_test=times_test,
        pdmd=pdmd_models[nl],                            # PDMD for this noise level
        dt_phys=dt_phys,
        Re_test=Re_test,
        mean_flow_train=mean_flow_noisy_dict[nl],        # mean flow for this noise level
        nu=nu,
        L_ref=L_ref
    )

